In [ ]:
# Databricks notebook sourceMAGIC %sqlselect * from bupa_bronze.providers

%md# Cell 1 — Imports & config

In [ ]:
MAGIC %run /Workspace/Repos/karthikvanapalli1505@gmail.com/Bupa_Insuarnce_Project/02_Silver_Layer/_00_utils_silver

In [ ]:
# ==============================================# PROVIDERS → SILVER# ==============================================from pyspark.sql import functions as Ffrom pyspark.sql.types import *DB_SILVER = "bupa_silver"spark.sql(f"CREATE DATABASE IF NOT EXISTS {DB_SILVER}")

%md# Cell 2 — Read bronze

In [ ]:
# If you have a Bronze tableprov_bz = spark.table("bupa_bronze.providers")# Otherwise, from ADLS path:# prov_bz = spark.read.format("delta").load(paths_bronze["providers"])print("Rows in Bronze Providers:", prov_bz.count())prov_bz.display()

%md# Cell 3 — Schema enforcement

In [ ]:
prov_schema = StructType([    StructField("Provider_ID", StringType()),    StructField("Fraud_Flag", IntegerType()),])prov = enforce_schema(prov_bz, prov_schema)prov.printSchema()

%md# Cell 4 — Key and data checks

In [ ]:
# ---------- Cell 4 — Key & Data Checks (type-safe normalization) ----------from pyspark.sql import functions as F# 1) Quarantine null Provider_IDprov_bad = prov.filter(F.col("Provider_ID").isNull())if prov_bad.take(1):    quarantine(prov_bad, "null_provider_id", "providers")prov_ok = prov.filter(F.col("Provider_ID").isNotNull())# 2) Normalize Provider_ID (upper/trim, allow A–Z, 0–9, underscore)prov_ok = prov_ok.withColumn(    "Provider_ID",    F.upper(F.regexp_replace(F.trim(F.col("Provider_ID")), r"[^A-Z0-9_]", "")))# 3) Normalize Fraud_Flag using STRING logic to avoid mixed typess = F.lower(F.trim(F.col("Fraud_Flag").cast("string")))prov_ok = prov_ok.withColumn(    "Fraud_Flag_norm",    F.when(s.isin("1","y","yes","true","t","on"), 1)     .when(s.isin("0","n","no","false","f","off"), 0)     .otherwise(None)  # invalid / unknown)# 4) Quarantine invalid/unknown flags; keep only valid 0/1bad_flag = prov_ok.filter(F.col("Fraud_Flag_norm").isNull())if bad_flag.take(1):    quarantine(bad_flag.drop("Fraud_Flag_norm"), "invalid_fraud_flag", "providers")prov_ok = (prov_ok    .filter(F.col("Fraud_Flag_norm").isNotNull())    .drop("Fraud_Flag")    .withColumnRenamed("Fraud_Flag_norm", "Fraud_Flag")    .withColumn("Fraud_Flag", F.col("Fraud_Flag").cast("int")))# (Optional) quick sanityprint("Rows after key/flag normalization:", prov_ok.count())

%md# Cell 5 — Deduplication

In [ ]:
# ---------- Cell 5 — Deduplication (robust, no empty-order bug) ----------from pyspark.sql import functions as Ffrom pyspark.sql.window import Window# We keep exactly one row per Provider_ID.# Prefer rows with Fraud_Flag = 1 (so any "high risk" signal survives),# then tie-break by Provider_ID (deterministic).if "Fraud_Flag" not in prov_ok.columns:    raise Exception("Expected column 'Fraud_Flag' missing after normalization in Cell 4.")w = Window.partitionBy("Provider_ID") \          .orderBy(              F.col("Fraud_Flag").desc_nulls_last(),              F.col("Provider_ID").asc()          )prov_silver = (prov_ok    .withColumn("_rn", F.row_number().over(w))    .filter(F.col("_rn") == 1)    .drop("_rn"))print("Rows after dedup:", prov_silver.count())# Optional peek:# display(prov_silver.limit(20))

%md# Cell 6 — Feature Engineering

In [ ]:
# Simple derived flag as textprov_silver = prov_silver.withColumn(    "Fraud_Risk_Label",    F.when(F.col("Fraud_Flag") == 1, "High Risk").otherwise("Normal"))

%md# Cell 7 — Data Quality Rules

In [ ]:
dq_expect(prov_silver, "key_not_null",          "Provider_ID IS NOT NULL",          "error", "providers", quarantine, "null_provider_id")dq_expect(prov_silver, "fraud_flag_valid",          "Fraud_Flag IN (0,1)",          "error", "providers", quarantine, "fraud_flag_invalid")

%md# Cell 8 — Write Silver

In [ ]:
full_table = f"{DB_SILVER}.providers"PATH_PROVIDERS = "abfss://silver@bupaprocesseddatastorage.dfs.core.windows.net/providers"(prov_silver.write    .format("delta")    .mode("overwrite")    .option("overwriteSchema", "true")    .saveAsTable(full_table))print(f"✅ Providers Silver written: {full_table}")# --- 2) Also save to external ADLS container (same data files)(prov_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(paths_silver["providers"]))print(f"✅ External Delta written to ADLS path:\n   {PATH_PROVIDERS}")try:    spark.sql(f"OPTIMIZE {full_table} ZORDER BY (Provider_ID)")except Exception as e:    print("[WARN] OPTIMIZE skipped:", e)

%md# Cell 9 — Data Quality Summary

In [ ]:
prov_summary = prov_silver.groupBy("Fraud_Flag","Fraud_Risk_Label").count().orderBy("Fraud_Flag")display(prov_summary)

%md# Cell 10 — Metrics

In [ ]:
from datetime import datetimespark.sql(f"""CREATE TABLE IF NOT EXISTS {DB_SILVER}.__run_metrics (  metric STRING, value STRING, context STRING, ts TIMESTAMP) USING DELTA""")def write_metric(n,v,c):    spark.createDataFrame([(n,str(v),c,datetime.utcnow())],        "metric STRING, value STRING, context STRING, ts TIMESTAMP") \      .write.mode("append").saveAsTable(f"{DB_SILVER}.__run_metrics")write_metric("rowcount_providers_silver", prov_silver.count(), "providers_silver")write_metric("distinct_provider_ids", prov_silver.select("Provider_ID").distinct().count(), "providers_silver")fraud_rate = prov_silver.filter(F.col("Fraud_Flag") == 1).count() / prov_silver.count()write_metric("fraud_rate_providers", round(fraud_rate,4), "providers_silver")print("📈 Metrics written.")

%md# Providers data before and after processing

In [ ]:
MAGIC %sqlselect * from bupa_bronze.providers

In [ ]:
MAGIC %sqlselect * from bupa_silver.providers

%md# Difference between bronze providers and silver providers

In [ ]:
# Load DataFramesbronze_df = spark.table("bupa_bronze.providers")silver_df = spark.table("bupa_silver.providers")bronze_cols = set(bronze_df.columns)silver_cols = set(silver_df.columns)# Identify new, dropped, and common featuresnew_features = silver_cols - bronze_colsdropped_features = bronze_cols - silver_colscommon_features = sorted(bronze_cols & silver_cols)print(f"New features added in silver: {sorted(new_features)}")print(f"Features dropped in silver: {sorted(dropped_features)}")# Get data types as dictsbronze_types = dict(bronze_df.dtypes)silver_types = dict(silver_df.dtypes)# Align on Provider_ID for accurate comparisonbronze_aligned = bronze_df.select("Provider_ID", *common_features).dropDuplicates(["Provider_ID"])silver_aligned = silver_df.select("Provider_ID", *common_features).dropDuplicates(["Provider_ID"])joined = (    bronze_aligned.alias("bz")    .join(silver_aligned.alias("sv"), on="Provider_ID", how="outer"))# For each common feature, check for changes and count them, and compare typesfor col in common_features:    bronze_type = bronze_types.get(col)    silver_type = silver_types.get(col)    type_changed = bronze_type != silver_type    type_msg = (        f"Type changed: {bronze_type} → {silver_type}"        if type_changed else        f"Type unchanged: {bronze_type}"    )    n_diff = (        joined        .filter(            (joined[f"bz.{col}"].isNull() & joined[f"sv.{col}"].isNotNull()) |            (joined[f"bz.{col}"].isNotNull() & joined[f"sv.{col}"].isNull()) |            (joined[f"bz.{col}"] != joined[f"sv.{col}"])        )        .count()    )    if n_diff == 0 and not type_changed:        print(f"Feature '{col}' is unchanged between bronze and silver. {type_msg}")    else:        print(f"Feature '{col}' was transformed between bronze and silver in {n_diff} providers. {type_msg}")        # Show a sample of differences        sample_diff = (            joined            .filter(                (joined[f"bz.{col}"].isNull() & joined[f"sv.{col}"].isNotNull()) |                (joined[f"bz.{col}"].isNotNull() & joined[f"sv.{col}"].isNull()) |                (joined[f"bz.{col}"] != joined[f"sv.{col}"])            )            .select("Provider_ID", f"bz.{col}", f"sv.{col}")            .limit(5)        )        print(f"Sample differences for '{col}':")        display(sample_diff)# Explain new features and show sample datafor col in sorted(new_features):    print(f"Feature '{col}' added: Reason unknown, please check transformation logic.")    print(f"Sample data for new feature '{col}':")    display(silver_df.select("Provider_ID", col).limit(5))# Note dropped featuresfor col in sorted(dropped_features):    print(f"Feature '{col}' was present in bronze but dropped in silver.")

%md# A) Snapshot basics: row counts & distinct IDs

In [ ]:
from pyspark.sql import functions as FBRONZE_TBL = "bupa_bronze.providers"   # adjust if differentSILVER_TBL = "bupa_silver.providers"br = spark.table(BRONZE_TBL)sv = spark.table(SILVER_TBL)metrics = {    "rows_bronze": br.count(),    "rows_silver": sv.count(),    "distinct_provider_id_bronze": br.select("Provider_ID").distinct().count(),    "distinct_provider_id_silver": sv.select("Provider_ID").distinct().count(),}print(metrics)

%md# B) Data quality improvement deltas(Null Provider_ID, Fraud_Flag validity, duplicate providers)

In [ ]:
from pyspark.sql import functions as Fdef pct(x, n):     return 0.0 if n == 0 else round(100.0 * x / n, 4)# ---- Bronze ----n_br = br.count()# Null keybr_null_key = br.filter(F.col("Provider_ID").isNull()).count()# Fraud_Flag validity in bronze (accept common variants)s_b = F.lower(F.trim(F.col("Fraud_Flag").cast("string")))br_invalid_flag = (br    .withColumn("_ok", F.when(s_b.isin("0","1","y","yes","n","no","true","false","t","f","on","off"), 1).otherwise(0))    .filter(F.col("_ok") == 0)    .count())# Duplicates by Provider_IDbr_dupes = n_br - br.select("Provider_ID").distinct().count()# ---- Silver ----n_sv = sv.count()sv_null_key = sv.filter(F.col("Provider_ID").isNull()).count()# Silver Fraud_Flag must be normalized to 0/1sv_invalid_flag = sv.filter(~F.col("Fraud_Flag").isin(0,1)).count()sv_dupes = n_sv - sv.select("Provider_ID").distinct().count()report = {    "bronze_null_provider_id_pct": pct(br_null_key, n_br),    "silver_null_provider_id_pct": pct(sv_null_key, n_sv),    "bronze_invalid_fraud_flag_pct": pct(br_invalid_flag, n_br),    "silver_invalid_fraud_flag_pct": pct(sv_invalid_flag, n_sv),    "bronze_duplicates_pct": pct(br_dupes, n_br),    "silver_duplicates_pct": pct(sv_dupes, n_sv),}report

%md# C) FK progress vs Claims (how well Claims.Provider_ID matches Providers)(Uses canonical IDs; shows improvement when using Silver providers vs Bronze providers.)

In [ ]:
from pyspark.sql import functions as F# Only run if Claims Silver existsCLAIMS_TBL = "bupa_silver.claims"claims = spark.table(CLAIMS_TBL)def canon(col):    return F.upper(F.regexp_replace(F.trim(F.col(col)), r"[^A-Z0-9_]", ""))claims_nonnull = (claims    .filter(F.col("Provider_ID").isNotNull())    .withColumn("Provider_ID_canon", canon("Provider_ID"))    .select("Claim_ID","Provider_ID_canon"))prov_bronze_canon = spark.table(BRONZE_TBL) \    .select(canon("Provider_ID").alias("Provider_ID_canon")).dropDuplicates()prov_silver_canon = spark.table(SILVER_TBL) \    .select(canon("Provider_ID").alias("Provider_ID_canon")).dropDuplicates()tot_claims_with_provider = claims_nonnull.count()miss_bronze = claims_nonnull.join(prov_bronze_canon, "Provider_ID_canon", "left_anti").count()miss_silver = claims_nonnull.join(prov_silver_canon, "Provider_ID_canon", "left_anti").count()def pct(x, n):     return 0.0 if n == 0 else round(100.0 * x / n, 4)fk_progress = {    "claims_with_provider_rows": tot_claims_with_provider,    "claims_fk_missing_pct_vs_bronze_providers": pct(miss_bronze, tot_claims_with_provider),    "claims_fk_missing_pct_vs_silver_providers": pct(miss_silver, tot_claims_with_provider)}fk_progress

%md# D) Feature coverage check (Silver-only columns that should exist)

In [ ]:
exists = {c: (c in sv.columns) for c in [    "Provider_ID",    "Fraud_Flag",    "Fraud_Risk_Label",  # derived in Cell 6 of your pipeline]}exists

%md# NotesNo references to Provider_Name, Region, Specialty, etc., because your current Providers pipeline (per your 10 cells) only carries Provider_ID, Fraud_Flag, and adds Fraud_Risk_Label.If you later enhance Providers Silver to include more attributes (name, specialty, region, accreditation), you can extend B and D to validate those too (dictionary checks, date ranges for contracts, etc.).

%md# Diagnostic

In [ ]:
from pyspark.sql import functions as Fdef canon(col):     return F.upper(F.regexp_replace(F.trim(F.col(col)), r"[^A-Z0-9_]", ""))# Distinct samples from each sideclaims = spark.table("bupa_silver.claims").filter(F.col("Provider_ID").isNotNull()) \           .withColumn("Provider_ID_canon", canon("Provider_ID"))prov   = spark.table("bupa_silver.providers") \           .withColumn("Provider_ID_canon", canon("Provider_ID"))print("claims providers distinct:", claims.select("Provider_ID_canon").distinct().count())print("providers distinct:",        prov.select("Provider_ID_canon").distinct().count())# Show a few claim provider IDs that don't exist in providersmiss = claims.select("Provider_ID_canon").distinct() \             .join(prov.select("Provider_ID_canon").distinct(), "Provider_ID_canon", "left_anti")display(miss.limit(25))# Compare patterns/lengthsdisplay(  claims.select(      F.length("Provider_ID_canon").alias("len"),      F.regexp_extract("Provider_ID_canon", r"^[A-Z_]+", 0).alias("prefix")  ).groupBy("len","prefix").count().orderBy(F.desc("count")))display(  prov.select(      F.length("Provider_ID_canon").alias("len"),      F.regexp_extract("Provider_ID_canon", r"^[A-Z_]+", 0).alias("prefix")  ).groupBy("len","prefix").count().orderBy(F.desc("count")))

In [ ]:
from pyspark.sql import functions as Fdef canon(c): return F.upper(F.regexp_replace(F.trim(F.col(c)), r"[^A-Z0-9_]", ""))claims = (spark.table("bupa_silver.claims")          .filter(F.col("Provider_ID").isNotNull())          .withColumn("Provider_ID_canon", canon("Provider_ID"))          .select("Provider_ID_canon").distinct())prov   = (spark.table("bupa_silver.providers")          .withColumn("Provider_ID_canon", canon("Provider_ID"))          .select("Provider_ID_canon").distinct())intersection = claims.join(prov, "Provider_ID_canon", "inner").count()only_in_claims = claims.join(prov, "Provider_ID_canon", "left_anti").count()only_in_prov   = prov.join(claims, "Provider_ID_canon", "left_anti").count()print({  "intersection": intersection,  "only_in_claims": only_in_claims,  "only_in_providers": only_in_prov})# show a few examples of claim provider ids that have no match in providersdisplay(claims.join(prov, "Provider_ID_canon", "left_anti").limit(25))